# Day 081 — Solution: A Multi-Tool Assistant

In [ ]:
_SRC = '"""tool_agent.py — Day 081: Tool-Using Agents.\n\nDays 79-80 gave an agent a couple of tools and let it call them. This day is\nabout tools *at scale*: a toolbox of many tools, each with a typed parameter\nschema; argument validation before anything runs; a first-class ToolRegistry;\nand tool *selection* - having the model route a request to the single best\ntool and extract its arguments.\n\nPieces (new on Day 081; helpers reused from Days 79-80):\n  safe_calculate / safe_parse_json / call_llm  - reused (Days 79-80)\n  DEFAULT_TOOLS            - a toolbox: each tool has a typed parameter schema\n  validate_args            - check args against a tool\'s schema before running\n  ToolRegistry             - register / get / describe / validate / execute\n  build_default_registry   - a ToolRegistry preloaded with DEFAULT_TOOLS\n  build_selection_prompt   - ask the model to pick one tool + its args\n  select_tool              - parse the model\'s choice (never raises)\n  route_query              - select -> validate -> execute\n  ToolAgent                - a multi-tool assistant over a registry\n\nSetup:\n    pip install ollama\n    ollama pull llama3.2\n"""\nimport ast\nimport json\nimport operator\n\n# ── helpers reused from Days 79-80 ───────────────────────────────────────────\n_OPS = {\n    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,\n    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,\n    ast.USub: operator.neg, ast.UAdd: operator.pos,\n}\n\n\ndef _eval_node(node):\n    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):\n        return node.value\n    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:\n        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))\n    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:\n        return _OPS[type(node.op)](_eval_node(node.operand))\n    raise ValueError("unsupported expression")\n\n\ndef safe_calculate(expression):\n    """Evaluate arithmetic without eval() (Day 79)."""\n    return _eval_node(ast.parse(expression, mode="eval").body)\n\n\ndef safe_parse_json(text):\n    """Slice first \'{\' to last \'}\' and parse. Returns dict|None (Day 79)."""\n    start, end = text.find("{"), text.rfind("}")\n    if start == -1 or end == -1 or end < start:\n        return None\n    try:\n        data = json.loads(text[start:end + 1])\n    except (json.JSONDecodeError, ValueError):\n        return None\n    return data if isinstance(data, dict) else None\n\n\ndef call_llm(messages, llm_fn=None):\n    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""\n    if llm_fn is not None:\n        return llm_fn(messages)\n    import ollama\n    resp = ollama.chat(model="llama3.2", messages=messages)\n    return resp["message"]["content"]\n\n# ── typed parameter schemas + argument validation ────────────────────────────\ndef _is_number(s):\n    try:\n        float(s)\n        return True\n    except (TypeError, ValueError):\n        return False\n\n\n# each check answers: does this value satisfy the declared type?\n_TYPE_CHECKS = {\n    "string": lambda v: isinstance(v, str),\n    "integer": lambda v: (isinstance(v, int) and not isinstance(v, bool))\n                         or (isinstance(v, str) and v.strip().lstrip("-").isdigit()),\n    "number": lambda v: (isinstance(v, (int, float)) and not isinstance(v, bool))\n                        or (isinstance(v, str) and _is_number(v)),\n    "boolean": lambda v: isinstance(v, bool),\n}\n\n\ndef validate_args(tool, args):\n    """Validate args against a tool\'s parameter schema.\n\n    Returns (ok: bool, error: str). Checks two things per declared parameter:\n    required parameters must be present, and present values must match the\n    declared type. An unknown declared type is treated as no constraint.\n    """\n    for pname, pspec in tool.get("parameters", {}).items():\n        if pspec.get("required") and pname not in args:\n            return False, "missing required parameter: " + pname\n        if pname in args:\n            check = _TYPE_CHECKS.get(pspec.get("type"))\n            if check is not None and not check(args[pname]):\n                return False, "parameter " + pname + " must be " + str(pspec.get("type"))\n    return True, ""\n\n\n# ── the toolbox: each tool declares name, description, typed schema, fn ───────\ndef _p(type_, required=True, description=""):\n    """Shorthand for a parameter spec."""\n    return {"type": type_, "required": required, "description": description}\n\n\nDEFAULT_TOOLS = [\n    {"name": "calculator",\n     "description": "Evaluate an arithmetic expression, e.g. 2 * (3 + 4).",\n     "parameters": {"expression": _p("string", description="the arithmetic")},\n     "fn": lambda args: str(safe_calculate(args["expression"]))},\n    {"name": "word_count",\n     "description": "Count the words in a piece of text.",\n     "parameters": {"text": _p("string", description="text to count")},\n     "fn": lambda args: str(len(str(args["text"]).split()))},\n    {"name": "uppercase",\n     "description": "Convert text to UPPERCASE.",\n     "parameters": {"text": _p("string", description="text to upcase")},\n     "fn": lambda args: str(args["text"]).upper()},\n    {"name": "reverse",\n     "description": "Reverse a piece of text.",\n     "parameters": {"text": _p("string", description="text to reverse")},\n     "fn": lambda args: str(args["text"])[::-1]},\n    {"name": "repeat",\n     "description": "Repeat a piece of text N times.",\n     "parameters": {"text": _p("string", description="text to repeat"),\n                    "times": _p("integer", description="how many times")},\n     "fn": lambda args: str(args["text"]) * int(args["times"])},\n]\n\n# ── the ToolRegistry ──────────────────────────────────────────────────────────\nclass ToolRegistry:\n    """A first-class collection of tools with validation and execution.\n\n    A registry owns its tools, renders them for a prompt, validates arguments\n    against each tool\'s schema, and executes safely (never raises).\n    """\n\n    def __init__(self, tools=None):\n        self._tools = {}\n        for tool in (tools or []):\n            self.register(tool)\n\n    def register(self, tool):\n        """Add a fully-formed tool dict; returns self."""\n        self._tools[tool["name"]] = tool\n        return self\n\n    def add(self, name, description, fn, parameters=None):\n        """Add a tool from parts; returns self."""\n        return self.register({"name": name, "description": description,\n                              "parameters": parameters or {}, "fn": fn})\n\n    def get(self, name):\n        return self._tools.get(name)\n\n    def names(self):\n        return list(self._tools)\n\n    def __contains__(self, name):\n        return name in self._tools\n\n    def __len__(self):\n        return len(self._tools)\n\n    def describe(self):\n        """Render the toolbox as prompt text: one line per tool."""\n        lines = []\n        for name, tool in self._tools.items():\n            params = ", ".join(tool.get("parameters", {}))\n            lines.append("- " + name + "(" + params + "): " + tool["description"])\n        return "\\n".join(lines)\n\n    def validate(self, name, args):\n        """Validate args for a named tool. Returns (ok, error)."""\n        tool = self.get(name)\n        if tool is None:\n            return False, "unknown tool: " + repr(name)\n        return validate_args(tool, args)\n\n    def execute(self, name, args):\n        """Validate then run a tool. Returns a result string; never raises."""\n        tool = self.get(name)\n        if tool is None:\n            return "Error: unknown tool " + repr(name)\n        ok, err = validate_args(tool, args)\n        if not ok:\n            return "Error: " + err\n        try:\n            return str(tool["fn"](args))\n        except Exception as exc:\n            return "Error running " + name + ": " + str(exc)\n\n\ndef build_default_registry():\n    """A ToolRegistry preloaded with the DEFAULT_TOOLS toolbox."""\n    return ToolRegistry(DEFAULT_TOOLS)\n\n# ── tool selection (routing) ──────────────────────────────────────────────────\ndef build_selection_prompt(query, registry):\n    """Ask the model to choose ONE tool for the request and extract its args."""\n    system = "\\n".join([\n        "You are a router. Choose the single best tool for the user request "\n        "and extract its arguments.",\n        "",\n        "Available tools:",\n        registry.describe(),\n        "",\n        "Reply with ONLY a JSON object:",\n        \'{"tool": "<tool name>", "args": {...}}\',\n        \'If no tool fits, reply {"tool": "none", "args": {}}.\',\n    ])\n    return [{"role": "system", "content": system},\n            {"role": "user", "content": "Request: " + str(query)}]\n\n\ndef select_tool(query, registry, llm_fn=None):\n    """Route a request to one tool. Returns {"tool": name, "args": dict}.\n\n    NEVER raises: unparseable output or an unknown tool name both fall back to\n    {"tool": "none", "args": {}}.\n    """\n    response = call_llm(build_selection_prompt(query, registry), llm_fn=llm_fn)\n    data = safe_parse_json(response) or {}\n    name = data.get("tool", "none")\n    args = data.get("args", {})\n    if name not in registry:\n        name = "none"\n    return {"tool": name, "args": args if isinstance(args, dict) else {}}\n\n# ── the router: select -> validate -> execute ────────────────────────────────\ndef route_query(query, registry, llm_fn=None):\n    """Select a tool for the query and run it.\n\n    Returns {"tool", "args", "result"}. If no tool fits, tool is "none" and no\n    tool runs. Validation and execution errors come back as the result string.\n    """\n    choice = select_tool(query, registry, llm_fn=llm_fn)\n    if choice["tool"] == "none":\n        return {"tool": "none", "args": {}, "result": "No suitable tool found."}\n    result = registry.execute(choice["tool"], choice["args"])\n    return {"tool": choice["tool"], "args": choice["args"], "result": result}\n\n# ── the multi-tool assistant ──────────────────────────────────────────────────\nclass ToolAgent:\n    """A multi-tool assistant: routes each request to the best tool.\n\n    Binds a ToolRegistry and an optional llm_fn, answers requests by routing,\n    and keeps a history of every ask.\n\n    Example::\n\n        agent = ToolAgent(llm_fn=my_llm_fn)\n        print(agent.ask("shout the word hello")["result"])\n    """\n\n    def __init__(self, registry=None, llm_fn=None):\n        self.registry = registry if registry is not None else build_default_registry()\n        self._llm_fn = llm_fn\n        self._history = []\n\n    def add_tool(self, name, description, fn, parameters=None):\n        """Register a new tool on this agent\'s registry; returns self."""\n        self.registry.add(name, description, fn, parameters)\n        return self\n\n    def tools(self):\n        """List the names of available tools."""\n        return self.registry.names()\n\n    def ask(self, query):\n        """Route one request to the best tool and run it. Returns the result dict."""\n        result = route_query(query, self.registry, llm_fn=self._llm_fn)\n        self._history.append({"query": query, "result": result})\n        return result\n\n    def history(self):\n        """Return a copy of the ask history."""\n        return list(self._history)\n\n    def clear_history(self):\n        """Clear the ask history in place."""\n        self._history.clear()\n'
from pathlib import Path
Path('tool_agent.py').write_text(_SRC, encoding='utf-8')
print('tool_agent.py written.')

In [ ]:

from tool_agent import (
    DEFAULT_TOOLS, validate_args, ToolRegistry, build_default_registry,
    build_selection_prompt, select_tool, route_query, ToolAgent,
)
import json

def _pick(tool, args):
    payload = json.dumps({'tool': tool, 'args': args})
    return lambda messages: payload

# 1. validate_args + toolbox
repeat = next(t for t in DEFAULT_TOOLS if t['name'] == 'repeat')
assert repeat['fn']({'text': 'ab', 'times': 3}) == 'ababab'
assert validate_args(repeat, {'text': 'hi', 'times': 2})[0] is True
assert validate_args(repeat, {'text': 'hi'})[0] is False            # missing
assert validate_args(repeat, {'text': 'hi', 'times': 'x'})[0] is False  # wrong type
print("✅ DEFAULT_TOOLS + validate_args")

# 2. ToolRegistry
reg = build_default_registry()
assert len(reg) >= 5 and 'calculator' in reg
assert reg.execute('uppercase', {'text': 'hi'}) == 'HI'
assert reg.execute('repeat', {'text': 'hi'}).lower().startswith('error')  # validation gate
assert 'repeat(' in reg.describe()
print("✅ ToolRegistry (describe / validate / execute)")

# 3. select_tool
msgs = build_selection_prompt('shout', reg)
assert 'calculator' in msgs[0]['content']
assert select_tool('x', reg, llm_fn=_pick('uppercase', {'text': 'hi'}))['tool'] == 'uppercase'
assert select_tool('x', reg, llm_fn=_pick('teleport', {}))['tool'] == 'none'  # hallucinated
assert select_tool('x', reg, llm_fn=lambda m: 'no idea')['tool'] == 'none'    # garbage
print("✅ select_tool (routing, fallback to none)")

# 4. route_query
r = route_query('x', reg, llm_fn=_pick('repeat', {'text': 'ab', 'times': 3}))
assert r['tool'] == 'repeat' and r['result'] == 'ababab'
assert route_query('x', reg, llm_fn=_pick('none', {}))['tool'] == 'none'
assert route_query('x', reg, llm_fn=_pick('repeat', {'text': 'ab'}))['result'].lower().startswith('error')
print("✅ route_query (select -> validate -> execute)")

# 5. ToolAgent
agent = ToolAgent(llm_fn=_pick('uppercase', {'text': 'hi'}))
assert agent.ask('shout hi')['result'] == 'HI'
assert 'calculator' in agent.tools() and 'repeat' in agent.tools()
agent.add_tool('double', 'Double a number.', lambda a: str(int(a['n']) * 2),
               {'n': {'type': 'integer', 'required': True}})
assert 'double' in agent.tools() and 'double' not in {t['name'] for t in DEFAULT_TOOLS}
assert len(agent.history()) == 1
agent.history().clear()
assert len(agent.history()) == 1     # history() returns a copy
agent.clear_history()
assert len(agent.history()) == 0
print("✅ ToolAgent (ask / add_tool / history / clear_history)")

print("\nMulti-tool assistant complete!")
